# 📂 Build Synthetic Knowledge Base

This notebook uses **`meta-llama/Llama-3.2-3B-Instruct`** via the HuggingFace Inference API
to generate a realistic synthetic knowledge base for **Insurellm**, an insurance tech company.

### What gets generated:
| Folder | Contents |
|--------|----------|
| `knowledge-base/company/` | About, overview, culture, careers pages |
| `knowledge-base/employees/` | HR-style profiles for each employee |
| `knowledge-base/products/` | Product spec sheets (features, pricing, roadmap) |
| `knowledge-base/contracts/` | Client contracts (one per product per client) |

### Requirements
- A free HuggingFace account + API token: [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
- Add `HF_TOKEN=hf_...` to your `.env` file
- Accept the model license at [meta-llama/Llama-3.2-3B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct)

### Customize it!
Change the `COMPANY` config in **Step 1** to generate a knowledge base for any company.

## Step 1 — Install Dependencies

In [ ]:
# Install huggingface_hub if not already installed
# !pip install huggingface_hub python-dotenv

## Step 2 — Setup & Company Config

Edit the `COMPANY` dictionary to define your company. Everything else is generated from this.

In [ ]:
import os
import random
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

load_dotenv(override=True)

# ── Model config ───────────────────────────────────────────────────────────────
MODEL_ID    = "meta-llama/Llama-3.2-3B-Instruct"
HF_TOKEN    = os.getenv("HF_TOKEN")   # set in .env as HF_TOKEN=hf_...
MAX_TOKENS  = 1500   # Llama-3.2-3B handles ~2K tokens output well
TEMPERATURE = 0.7

client = InferenceClient(model=MODEL_ID, token=HF_TOKEN)

print(f"✅ Using model : {MODEL_ID}")
print(f"✅ HF token    : {'set ✓' if HF_TOKEN else '❌ NOT SET — add HF_TOKEN to .env'}")

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT          = Path("..").resolve()
KB            = ROOT / "knowledge-base"
COMPANY_DIR   = KB / "company"
EMPLOYEES_DIR = KB / "employees"
PRODUCTS_DIR  = KB / "products"
CONTRACTS_DIR = KB / "contracts"

for d in [COMPANY_DIR, EMPLOYEES_DIR, PRODUCTS_DIR, CONTRACTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Company config — edit this! ────────────────────────────────────────────────
COMPANY = {
    "name":        "Insurellm",
    "founded":     2015,
    "founder":     "Avery Lancaster",
    "industry":    "Insurance Technology (InsurTech)",
    "hq":          "San Francisco, California",
    "description": "An AI-powered InsurTech company that sells software platforms to insurance carriers, enabling them to offer better products at lower cost.",
    "employees":   32,
    "products": [
        {"name": "Carllm",    "type": "Auto Insurance Platform"},
        {"name": "Homellm",   "type": "Home Insurance Platform"},
        {"name": "Healthllm", "type": "Health Insurance Platform"},
        {"name": "Lifellm",   "type": "Life Insurance Platform"},
        {"name": "Bizllm",    "type": "Commercial/Business Insurance Platform"},
        {"name": "Claimllm",  "type": "Claims Processing Platform"},
        {"name": "Rellm",     "type": "Reinsurance Platform"},
        {"name": "Markellm",  "type": "Insurance Marketplace Platform"},
    ],
    "employee_roles": [
        {"name": "Avery Lancaster", "title": "Co-Founder & CEO",          "dept": "Executive"},
        {"name": "Marcus Johnson",  "title": "Chief Technology Officer",  "dept": "Engineering"},
        {"name": "Sarah Williams",  "title": "VP of Sales",              "dept": "Sales"},
        {"name": "Rachel Martinez", "title": "Head of Product",          "dept": "Product"},
        {"name": "Daniel Park",     "title": "Senior Software Engineer", "dept": "Engineering"},
        {"name": "Emily Carter",    "title": "Data Scientist",           "dept": "Data"},
        {"name": "Kevin Zhang",     "title": "DevOps Engineer",          "dept": "Engineering"},
        {"name": "Lisa Anderson",   "title": "Customer Success Manager", "dept": "Customer Success"},
        {"name": "Jordan Blake",    "title": "UX Designer",             "dept": "Design"},
        {"name": "Nina Patel",      "title": "Marketing Manager",       "dept": "Marketing"},
    ],
    "clients_per_product": 2,   # number of contracts to generate per product
}

print(f"\n✅ Config ready for : {COMPANY['name']}")
print(f"   Products          : {len(COMPANY['products'])}")
print(f"   Employees         : {len(COMPANY['employee_roles'])}")
print(f"   Contracts         : {len(COMPANY['products']) * COMPANY['clients_per_product']} total")

## Step 3 — Helper: Call LLM & Save File

`InferenceClient.chat.completions.create()` uses the **same OpenAI-compatible interface**,
so the call pattern is identical — just a different model behind it.

In [ ]:
def llm(prompt: str, system: str = "You are a helpful business document writer. Write realistic, detailed content.") -> str:
    """
    Call meta-llama/Llama-3.2-3B-Instruct via HuggingFace Inference API.
    Uses the same OpenAI-compatible chat completions interface.
    """
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt},
        ],
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
    )
    return response.choices[0].message.content


def save(path: Path, content: str):
    """Write content to a .md file and print confirmation."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
    print(f"  ✅ Saved: {path.relative_to(ROOT)}")


# ── Quick sanity check ─────────────────────────────────────────────────────────
print("🧪 Testing model connection...")
test = llm("Say 'Hello from Llama!' and nothing else.")
print(f"   Model response: {test.strip()}")
print("✅ Model is working!")

## Step 4 — Generate Company Pages

Creates 4 company-level documents:
- `about.md` — founding story, mission, growth  
- `overview.md` — products, market position, financials  
- `culture.md` — values, team culture, remote work policy  
- `careers.md` — hiring, benefits, open roles

In [ ]:
c = COMPANY  # shorthand
product_list = ", ".join(p["name"] for p in c["products"])

company_pages = {
    "about.md": f"""
Write a detailed 'About Us' page for {c['name']}, an {c['industry']} company.
Founded in {c['founded']} by {c['founder']} in {c['hq']}.
Description: {c['description']}
Products: {product_list}
Current employee count: {c['employees']}

Include: founding story, mission, product history, growth milestones, current state.
Write in markdown with headers. Be specific with dates, numbers, and facts. About 400 words.
Do not add any commentary before or after the markdown document.
""",
    "overview.md": f"""
Write a company overview document for {c['name']}.
Industry: {c['industry']}. HQ: {c['hq']}. Founded: {c['founded']}.
Products: {product_list}

Include: executive summary, product portfolio summary with brief descriptions,
market position, key metrics (contracts, revenue range, client count), and strategic priorities.
Write in markdown with headers. Be specific with plausible business numbers. About 400 words.
Do not add any commentary before or after the markdown document.
""",
    "culture.md": f"""
Write a culture and values document for {c['name']}.
The company is a lean, remote-first {c['industry']} startup with {c['employees']} employees.

Include: core values (at least 5), remote work policy, team rituals, DEI commitments,
work-life balance philosophy, learning and development programs.
Write in markdown with headers. Tone should be authentic and specific. About 350 words.
Do not add any commentary before or after the markdown document.
""",
    "careers.md": f"""
Write a careers page for {c['name']}.
The company has {c['employees']} employees and is a remote-first {c['industry']} company.

Include: why join us, benefits package (salary, equity, health, PTO, L&D budget),
hiring process steps, open roles (at least 5 real-sounding roles with descriptions),
and one employee testimonial.
Write in markdown with headers. Be specific and compelling. About 400 words.
Do not add any commentary before or after the markdown document.
""",
}

print(f"🏢 Generating company pages for {c['name']}...\n")
for filename, prompt in company_pages.items():
    print(f"  📝 {filename}...")
    content = llm(prompt)
    save(COMPANY_DIR / filename, content)

print("\n✅ All company pages generated!")

## Step 5 — Generate Employee Profiles

Each employee gets a full HR-style profile with:
- Summary (DOB, title, location, salary)
- Career progression (at the company + prior roles)
- Annual performance history
- Compensation history
- HR notes

> 💡 **Llama-3.2-3B tip**: We keep prompts focused and explicit so the smaller model stays on-task.

In [ ]:
EMPLOYEE_SYSTEM = (
    "You are an HR document writer for a tech company. "
    "Write realistic, detailed employee profiles in markdown format only. "
    "Do not add any commentary, preamble, or explanation outside the markdown."
)

def employee_prompt(emp):
    joined_year = random.randint(2015, 2022)
    years_range = " → ".join(str(y) for y in range(joined_year, 2026))
    return f"""
Write an HR employee profile for {emp['name']} at {c['name']}.
Title: {emp['title']}. Department: {emp['dept']}. Joined: {joined_year}.

Use EXACTLY these markdown sections:

# {emp['name']}

## Summary
- Date of Birth: (realistic for career stage)
- Job Title: {emp['title']}
- Location: (one of San Francisco, New York, Austin, Chicago, Denver, or Remote)
- Current Salary: (realistic for {emp['title']})

## {c['name']} Career Progression
- {joined_year} - Present: {emp['title']} at {c['name']} (describe key achievements)
- (2-3 previous roles at other companies before {c['name']})

## Annual Performance History
One bullet per year ({years_range}).
Ratings must be one of: Exceptional / Exceeds Expectations / Meets Expectations / Satisfactory / Developing / Below Expectations
Format: - **YEAR**: **Rating** — one sentence reason.

## Compensation History
One bullet per year from {joined_year} to 2025.
Format: - **YEAR**: $X base + $Y bonus

## Other HR Notes
- Professional development
- Work-life balance
- Community engagement
"""

print(f"👥 Generating {len(c['employee_roles'])} employee profiles...\n")
for emp in c['employee_roles']:
    print(f"  📝 {emp['name']} ({emp['title']})...")
    content = llm(employee_prompt(emp), system=EMPLOYEE_SYSTEM)
    save(EMPLOYEES_DIR / f"{emp['name']}.md", content)

print(f"\n✅ All employee profiles generated!")

## Step 6 — Generate Product Spec Sheets

Each product gets a detailed spec sheet with:
- Summary (what it does, who it's for)
- Key features (7+)
- Pricing tiers (Basic / Professional / Enterprise)
- 2025-2026 product roadmap

In [ ]:
PRODUCT_SYSTEM = (
    "You are a B2B SaaS product marketing writer. "
    "Write detailed, realistic product specifications in markdown only. "
    "Do not add any commentary or preamble outside the markdown."
)

def product_prompt(product):
    return f"""
Write a product spec sheet for '{product['name']}' by {c['name']}.
Product type: {product['type']}
Company: {c['name']} — {c['description']}

Use EXACTLY these markdown sections:

# Product Summary
# {product['name']}

## Summary
2-3 paragraphs: what the product does, who it serves (B2B insurance carriers),
and what makes it unique. Mention AI and machine learning capabilities.

## Features
List at least 7 features. Each must have a bold feature name followed by a 1-2 sentence description.
Features must be specific to {product['type']}.

## Pricing
- **Basic Tier**: $X/month — list 3-4 inclusions
- **Professional Tier**: $Y/month — list 4-5 inclusions
- **Enterprise Tier**: $Z/month — list 5-6 inclusions
Include a contact-sales note for custom pricing.

## 2025-2026 Roadmap
### Q1 2025: (title)
- 2-3 bullet features
### Q2 2025: (title)
- 2-3 bullet features
### Q3 2025: (title)
- 2-3 bullet features
### Q4 2025: (title)
- 2-3 bullet features
### 2026: (annual goals)
- 2-3 bullet features
"""

print(f"📦 Generating {len(c['products'])} product spec sheets...\n")
for product in c['products']:
    print(f"  📝 {product['name']} ({product['type']})...")
    content = llm(product_prompt(product), system=PRODUCT_SYSTEM)
    save(PRODUCTS_DIR / f"{product['name']}.md", content)

print(f"\n✅ All product spec sheets generated!")

## Step 7 — Generate Client Contracts

For each product, generates `clients_per_product` contracts with fictional insurance companies.

Each contract includes:
- Contract number, date, parties
- Terms (coverage, duration, payment schedule)
- Features included
- SLA + support package
- Signatures

In [ ]:
CONTRACT_SYSTEM = (
    "You are a legal and business document writer specializing in B2B SaaS contracts. "
    "Write detailed, realistic contracts in markdown only. "
    "Do not add any commentary or explanation outside the markdown."
)

# Pool of fictional insurance client companies
CLIENT_POOL = [
    {"name": "DriveSmart Insurance Corp.",    "contact": "Steven Brooks",   "title": "President & COO"},
    {"name": "Velocity Auto Solutions Inc.",   "contact": "Patricia Cole",   "title": "CEO"},
    {"name": "TechDrive Insurance Group",       "contact": "Alan Torres",     "title": "Managing Director"},
    {"name": "Roadway Insurance Inc.",          "contact": "Catherine Moore", "title": "VP Operations"},
    {"name": "SafeHaven Property Insurance",    "contact": "Robert Ellis",    "title": "CEO"},
    {"name": "Greenstone Insurance Co.",        "contact": "Laura Simmons",   "title": "President"},
    {"name": "Pinnacle Insurance Co.",          "contact": "James Porter",    "title": "COO"},
    {"name": "GreenValley Insurance",           "contact": "Melissa Grant",   "title": "CEO"},
    {"name": "Harmony Health Plans",            "contact": "Dr. Sandra Obi",  "title": "President"},
    {"name": "WellCare Insurance Co.",          "contact": "Michael Tang",    "title": "CEO"},
    {"name": "United Healthcare Alliance",      "contact": "Denise Crawford", "title": "EVP"},
    {"name": "Advantage Medical Coverage",      "contact": "Thomas Webb",     "title": "Managing Director"},
    {"name": "Heritage Life Assurance",         "contact": "Grace Chen",      "title": "CEO"},
    {"name": "Guardian Life Partners",          "contact": "William Ford",    "title": "President"},
    {"name": "Evergreen Life Insurance",        "contact": "Nancy Walsh",     "title": "COO"},
    {"name": "Metropolitan Life Group",         "contact": "Charles King",    "title": "EVP Strategic Accounts"},
    {"name": "Atlantic Risk Solutions",         "contact": "Diana Ross",      "title": "CEO"},
    {"name": "Continental Commercial Group",    "contact": "Frank Nguyen",    "title": "President"},
    {"name": "Summit Commercial Insurance",     "contact": "Helen Park",      "title": "Managing Director"},
    {"name": "Fortress Business Underwriters",  "contact": "Edward Kim",      "title": "CEO"},
    {"name": "Premier Adjusters Inc.",          "contact": "Gloria Reyes",    "title": "President"},
    {"name": "FastTrack Insurance Services",    "contact": "Victor Marsh",    "title": "COO"},
    {"name": "National Claims Network",         "contact": "Tiffany Scott",   "title": "CEO"},
    {"name": "Rapid Claims Associates",         "contact": "Keith Barnes",    "title": "Managing Director"},
    {"name": "GlobalRe Partners",               "contact": "Irene Volkov",    "title": "CEO"},
    {"name": "Apex Reinsurance Group",          "contact": "Samuel Adeyemi",  "title": "President"},
    {"name": "EverGuard Insurance",             "contact": "Brenda Flores",   "title": "COO"},
    {"name": "Stellar Insurance Co.",           "contact": "Timothy Holt",    "title": "CEO"},
    {"name": "ConnectInsure Agency",            "contact": "Olivia Chang",    "title": "President"},
    {"name": "GreenField Holdings",             "contact": "Marcus Webb",     "title": "Managing Director"},
    {"name": "BrightWay Solutions",             "contact": "Sophia Laurent",  "title": "CEO"},
    {"name": "Belvedere Insurance",             "contact": "Andrew Novak",    "title": "COO"},
]

random.shuffle(CLIENT_POOL)

def contract_prompt(product, client, contract_num, year, tier, months):
    return f"""
Write a B2B SaaS contract between {c['name']} and {client['name']}.

Use EXACTLY this markdown structure:

# Contract with {client['name']} for {product['name']}

**Contract Date:** (realistic date in {year})
**Contract Number:** {contract_num}
**Parties:**
- {c['name']}, Inc.
- {client['name']}

---

## Terms
Number each term 1-7:
1. Coverage: what {client['name']} gets ({tier} Tier of {product['name']})
2. Duration: {months} months from contract date
3. Payment: specific monthly fee and total contract value (realistic for {tier} tier of a {product['type']} SaaS)
4. Policy Volume: number of policies/users {client['name']} manages
5. Service Level Agreement: uptime % and credit terms for downtime
6. Confidentiality: standard NDA clause
7. Data Security: compliance standards (SOC 2, GDPR, etc.)

---

## Renewal
Notice period, renewal options, pricing guarantee.

---

## Features
List 5-7 specific features of the {tier} Tier of {product['name']} that {client['name']} receives.
Use numbered list. Be specific to {product['type']}.

---

## Support
1. Dedicated success team details
2. Response time SLAs by priority (P1/P2/P3)
3. Implementation timeline
4. Training program

---

**Signatures:**

_________________________________
**{c['founder']}**
**Title**: CEO
**{c['name']}, Inc.**
**Date**: (same as contract date)

_________________________________
**{client['contact']}**
**Title**: {client['title']}
**{client['name']}**
**Date**: (same as contract date)
"""

n_contracts = COMPANY['clients_per_product'] * len(c['products'])
print(f"📝 Generating {n_contracts} contracts ({COMPANY['clients_per_product']} per product × {len(c['products'])} products)...\n")

client_idx   = 0
contract_idx = 1
years        = [2023, 2024, 2025]
tiers        = ["Basic", "Professional", "Enterprise"]
durations    = [12, 24, 36]

for product in c['products']:
    for _ in range(c['clients_per_product']):
        client       = CLIENT_POOL[client_idx % len(CLIENT_POOL)]
        year         = random.choice(years)
        tier         = random.choice(tiers)
        months       = random.choice(durations)
        contract_num = f"CR-{year}-{chr(65 + (contract_idx % 26))}-{contract_idx:04d}"

        print(f"  📝 {client['name']} × {product['name']} ({tier}, {months}mo)...")
        content  = llm(contract_prompt(product, client, contract_num, year, tier, months),
                       system=CONTRACT_SYSTEM)
        filename = f"Contract with {client['name']} for {product['name']}.md"
        save(CONTRACTS_DIR / filename, content)

        client_idx   += 1
        contract_idx += 1

print(f"\n✅ All contracts generated!")

## Step 8 — Summary

Count what was generated and view the full knowledge base structure.

In [ ]:
KB = Path("..").resolve() / "knowledge-base"

print("📁 Knowledge Base Summary")
print("=" * 45)

total_files = 0
total_bytes = 0
for folder in sorted(KB.iterdir()):
    if folder.is_dir():
        files = list(folder.rglob("*.md"))
        total_files += len(files)
        folder_bytes = sum(f.stat().st_size for f in files)
        total_bytes  += folder_bytes
        print(f"\n📂 {folder.name}/ ({len(files)} files, {folder_bytes/1024:.1f} KB)")
        for f in sorted(files):
            print(f"   • {f.name}")

print(f"\n{'=' * 45}")
print(f"Total : {total_files} files | {total_bytes/1024:.1f} KB")
print(f"\n🎉 Knowledge base is ready!")
print(f"   ✅ Step 1: Knowledge base built (this notebook)")
print(f"   → Step 2: Open notebooks/02_rag_implementations.ipynb")

## Step 9 — Preview Sample Output

Read the first file from each category to verify output quality.

In [ ]:
KB = Path("..").resolve() / "knowledge-base"

for folder_name in ["company", "employees", "products", "contracts"]:
    folder = KB / folder_name
    files  = sorted(f for f in folder.glob("*.md") if f.name != ".gitkeep")
    if not files:
        print(f"\n⚠️  No files in {folder_name}/")
        continue
    sample  = files[0]
    content = sample.read_text(encoding="utf-8")
    print(f"\n{'═' * 60}")
    print(f"📄 {folder_name}/{sample.name}")
    print(f"{'═' * 60}")
    print(content[:700])
    if len(content) > 700:
        print(f"\n... ({len(content)-700} more chars)")